# Identifiability Metric Deep-Dive
**Judge:** `llama-3.3-70b` | **Base LLM:** `gemma-3-27b-it`

Analysis of how easily users can recognize ad content across different injection methods.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

XLSX_PATH = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/output/evaluate_results/output/20260928_082937_judge_llama-3.3-70b-instruct-fp8_base_gemma-3-27b-it/identifiability_evaluation.xlsx'

df = pd.read_excel(XLSX_PATH, skiprows=1, header=None)
df.columns = ['question', 'method', 'response', 'analysis', 'category', 'score']

# Map categories to ordered labels
CAT_ORDER = ['NOT_IDENTIFIABLE', 'SOMEWHAT_IDENTIFIABLE', 'IDENTIFIABLE', 'HIGHLY_IDENTIFIABLE']
CAT_LABELS = {'NOT_IDENTIFIABLE': 'Not Identifiable', 'SOMEWHAT_IDENTIFIABLE': 'Somewhat Identifiable',
              'IDENTIFIABLE': 'Identifiable', 'HIGHLY_IDENTIFIABLE': 'Highly Identifiable'}

METHOD_ORDER = ['Ad-Chat', 'GI-R', 'GI-P', 'GIR-R', 'GIR-P']

print(f"Total rows: {len(df)}")
print(f"Methods: {df['method'].unique()}")
print(f"Categories: {df['category'].unique()}")
print(f"Score range: {df['score'].min()} - {df['score'].max()}")


Total rows: 3450
Methods: <StringArray>
['Ad-Chat', 'GI-R', 'GI-P', 'GIR-R', 'GIR-P']
Length: 5, dtype: str
Categories: <StringArray>
['SOMEWHAT_IDENTIFIABLE',   'HIGHLY_IDENTIFIABLE',          'IDENTIFIABLE',
      'NOT_IDENTIFIABLE']
Length: 4, dtype: str
Score range: 0 - 100


## 1. Overall Score Distribution by Method

In [2]:
# Mean, median, std per method
stats = df.groupby('method')['score'].agg(['mean', 'median', 'std', 'min', 'max']).reindex(METHOD_ORDER).round(2)
stats.columns = ['Mean', 'Median', 'Std', 'Min', 'Max']
display(stats)


,Mean,Median,Std,Min,Max
method,,,,,
Ad-Chat,36.56,33.0,12.69,0,100
GI-R,43.62,33.0,21.73,0,100
GI-P,42.50,33.0,20.28,0,100
GIR-R,40.29,33.0,18.68,0,100
GIR-P,38.25,33.0,15.65,0,100


## 2. Category Distribution by Method
Percentage of responses falling into each identifiability category.

In [3]:
# Crosstab: method x category (percentages)
ct = pd.crosstab(df['method'], df['category'], normalize='index').reindex(METHOD_ORDER)[CAT_ORDER] * 100
ct.columns = [CAT_LABELS[c] for c in ct.columns]
ct = ct.round(2)
display(ct)

print("\nAbsolute counts:")
ct_abs = pd.crosstab(df['method'], df['category']).reindex(METHOD_ORDER)[CAT_ORDER]
ct_abs.columns = [CAT_LABELS[c] for c in ct_abs.columns]
display(ct_abs)


,Not Identifiable,Somewhat Identifiable,Identifiable,Highly Identifiable
method,,,,
Ad-Chat,0.14,91.30,6.23,2.32
GI-R,0.43,77.10,12.61,9.86
GI-P,0.87,77.10,14.64,7.39
GIR-R,0.72,83.04,9.86,6.38
GIR-P,0.29,87.68,7.97,4.06



Absolute counts:


,Not Identifiable,Somewhat Identifiable,Identifiable,Highly Identifiable
method,,,,
Ad-Chat,1,630,43,16
GI-R,3,532,87,68
GI-P,6,532,101,51
GIR-R,5,573,68,44
GIR-P,2,605,55,28


## 3. "Highly Identifiable" Rate by Method
Which methods are most/least obvious to users?

In [4]:
# Highly identifiable rate
hi_rate = df.groupby('method').apply(lambda x: (x['category'] == 'HIGHLY_IDENTIFIABLE').mean() * 100).reindex(METHOD_ORDER).round(2)
hi_df = pd.DataFrame({'Method': METHOD_ORDER, 'Highly Identifiable %': hi_rate.values})
display(hi_df)

# Not identifiable rate
ni_rate = df.groupby('method').apply(lambda x: (x['category'] == 'NOT_IDENTIFIABLE').mean() * 100).reindex(METHOD_ORDER).round(2)
ni_df = pd.DataFrame({'Method': METHOD_ORDER, 'Not Identifiable %': ni_rate.values})
print("\nNot Identifiable rate:")
display(ni_df)


,Method,Highly Identifiable %
0,Ad-Chat,2.32
1,GI-R,9.86
2,GI-P,7.39
3,GIR-R,6.38
4,GIR-P,4.06



Not Identifiable rate:


,Method,Not Identifiable %
0,Ad-Chat,0.14
1,GI-R,0.43
2,GI-P,0.87
3,GIR-R,0.72
4,GIR-P,0.29
